## Day 10 — Query Optimization & Explain Plans

**What we learn:** How Spark executes your queries — and how to make them faster.

**Three tools:**
- `.explain('formatted')` — read the query plan before running
- **Partition pruning** — Spark skips entire file folders using your filter
- **Caching** — store a table in RAM so repeated queries skip disk entirely

**Tables used:** `ecommerce.silver.events_clean` and `ecommerce.gold.user_predictions`

---
**How to read an explain plan — bottom to top:**
```
HashAggregate          <- 3. final result
  Exchange             <- 2. shuffle across workers (expensive)
    FileScan parquet   <- 1. start here — reading files from disk
```
`FileScan` = slow (disk). `InMemoryTableScan` = fast (cache). `Exchange` = shuffle (costly).

In [0]:
import time
from pyspark.sql.functions import col, count, avg, round, desc, sum as spark_sum

CATALOG = 'ecommerce'
print('Ready')


### Task 1 — Run a Heavy Query (baseline)

Run a real aggregation on the full events table and time it. This is the **unoptimized baseline** you will compare against after caching.

Record `t1` — you will use it in Cell 7.

In [0]:
# Task 1 — Run a heavy aggregation and record the time
# groupBy + agg on 5.7M rows — reads full table from disk

start = time.time()

df_baseline = (
    spark.table(f'{CATALOG}.silver.events_clean')
    .groupBy('event_type')
    .agg(
        count('*').alias('event_count'),
        round(avg('price'), 2).alias('avg_price')
    )
    .orderBy(desc('event_count'))
)
df_baseline.show()

t1 = round(time.time() - start, 2)
print(f'Baseline time (no cache): {t1}s')
print('Remember this number — you will compare it after caching in Cell 7')


### Task 2 — Read the Explain Plan

`.explain('formatted')` prints the execution plan — what Spark will do **before it touches a single row**.

**What to look for:**
- `FileScan` at the bottom → reads raw files from Delta (slow, expected before caching)
- `Exchange hashpartitioning` → shuffle between workers (expensive but necessary for groupBy)
- `PartitionFilters` → partition pruning is active (good — only reads matching folders)
- `PushedFilters` → WHERE clause applied at file level (good — less data read per file)

In [0]:
# Task 2 — Explain plan BEFORE caching
# Read bottom to top

events = spark.table(f'{CATALOG}.silver.events_clean')

query = (
    events
    .groupBy('event_type')
    .agg(
        count('*').alias('event_count'),
        round(avg('price'), 2).alias('avg_price')
    )
)

print('=== EXPLAIN PLAN — BEFORE CACHING ===')
print('Read from BOTTOM to TOP')
print('Bottom = where Spark starts (file read)')
print('Top    = final output')
print()
query.explain('formatted')

# What you should see:
# - FileScan at the bottom       -> reading raw parquet files (no cache)
# - Exchange hashpartitioning    -> shuffle to group same event_types
# - HashAggregate x2             -> partial agg per partition, then final agg
# - NO InMemoryTableScan         -> expected, table not cached yet


In [0]:
# Cell 4 — Plain English explanation of what you saw

print('=== WHAT THE PLAN MEANS ===')
print()
print('FileScan parquet (bottom):')
print('  Spark is reading raw .parquet files from your Delta table storage')
print('  Every time you run this query it goes back to disk')
print()
print('Exchange hashpartitioning:')
print('  Spark shuffles rows across workers so all rows with the same')
print('  event_type end up on the same worker for counting')
print('  This is unavoidable for a groupBy — but it is the expensive part')
print()
print('HashAggregate (appears twice):')
print('  First: partial count on each worker (before shuffle)')
print('  Second: combine partial counts after shuffle')
print()
print('After caching: FileScan -> InMemoryTableScan (reads from RAM not disk)')
print('The Exchange will still be there — caching helps the read, not the shuffle')


### Partition Pruning Demo

Filter on the partition column and compare the plan with an unfiltered query. If `events_clean` is partitioned by `event_type`, the filtered plan will show `PartitionFilters` and skip the other folders entirely.

In [0]:
# Partition pruning — filter on partitioned column = Spark skips entire folders

events = spark.table(f'{CATALOG}.silver.events_clean')

# Query A: no filter — Spark scans ALL partitions
print('=== PLAN A: no filter (scans all partitions) ===')
events.groupBy('event_type').count().explain('formatted')

# Query B: filter on event_type — Spark only opens purchase/ folder
print('=== PLAN B: filter on event_type == purchase ===')
print('Look for: PartitionFilters in the plan — absent in Plan A')
events.filter(col('event_type') == 'purchase').explain('formatted')

# Count to show how many rows were actually read
purchase_count = events.filter(col('event_type') == 'purchase').count()
total_count    = events.count()
print(f'Purchase rows : {purchase_count:,}')
print(f'Total rows    : {total_count:,}')
print(f'Pruning saved : {total_count - purchase_count:,} rows not scanned')


### Task 3 — Enable Caching

`spark.catalog.cacheTable()` is the serverless-safe way to cache. Important: Spark is **lazy** — `cacheTable()` alone doesn't build the cache. You must run an action (like `.count()`) to force it to materialise.

In [0]:
# Task 3 — Cache the events table
# spark.catalog.cacheTable() is serverless-safe
# df.cache() may not work depending on serverless memory limits

print('Caching events_clean...')
spark.catalog.cacheTable(f'{CATALOG}.silver.events_clean')

# Force cache to build — Spark is lazy, cacheTable() alone does nothing
# The first .count() reads from disk AND stores in memory
# All subsequent queries will read from memory
_ = spark.table(f'{CATALOG}.silver.events_clean').count()

is_cached = spark.catalog.isCached(f'{CATALOG}.silver.events_clean')
print(f'Cached: {is_cached}')
print()

# Check the explain plan — FileScan should now be InMemoryTableScan
print('=== EXPLAIN PLAN — AFTER CACHING ===')
print('FileScan should now say InMemoryTableScan')
print()
(
    spark.table(f'{CATALOG}.silver.events_clean')
    .groupBy('event_type')
    .agg(count('*').alias('n'))
    .explain('formatted')
)


### Task 4 — Compare Execution Time

Re-run the same query from Cell 2 with the cache now active. Compare `t2` (cached) against `t1` (no cache). Expect the cached query to be noticeably faster — it reads from RAM instead of disk.

In [0]:
# Task 4 — Same query as Cell 2, now with cache active

start = time.time()

df_cached = (
    spark.table(f'{CATALOG}.silver.events_clean')
    .groupBy('event_type')
    .agg(
        count('*').alias('event_count'),
        round(avg('price'), 2).alias('avg_price')
    )
    .orderBy(desc('event_count'))
)
df_cached.show()

t2 = round(time.time() - start, 2)

print('=== EXECUTION TIME COMPARISON ===')
print(f'Without cache : {t1}s')
print(f'With cache    : {t2}s')
speedup = round(t1 / t2, 1) if t2 > 0 else 'N/A'
print(f'Speedup       : {speedup}x faster')
print()
print('Note: if speedup is small, the query is shuffle-bound not IO-bound')
print('Caching removes disk reads — the Exchange (shuffle) still happens')


In [0]:
# Run the same query 3 times and time each
# Run 1 = cold  (reads from disk, stores in cache)
# Run 2 = warm  (reads from cache)
# Run 3 = warm  (reads from cache)
# Shows the consistent speedup after the first run

# Clear cache first so Run 1 is truly cold
spark.catalog.clearCache()
spark.catalog.cacheTable(f'{CATALOG}.silver.events_clean')

times = []
for run in range(1, 4):
    start = time.time()
    spark.table(f'{CATALOG}.silver.events_clean') \
        .groupBy('event_type') \
        .agg(count('*').alias('n')) \
        .collect()
    elapsed = round(time.time() - start, 2)
    times.append(elapsed)
    label = 'cold (disk + build cache)' if run == 1 else 'warm (from RAM)'
    print(f'Run {run}: {elapsed}s  <- {label}')

print()
if times[2] > 0:
    print(f'Cache speedup (run 1 vs run 3): {round(times[0]/times[2],1)}x')


In [0]:
# Final cell: show before and after explain plan cleanly

q = (
    spark.table(f'{CATALOG}.silver.events_clean')
    .groupBy('event_type')
    .count()
)

# BEFORE: uncache so plan shows FileScan
spark.catalog.uncacheTable(f'{CATALOG}.silver.events_clean')
print('=== BEFORE CACHING — reads from disk ===')
print('Look for: FileScan at the bottom')
q.explain('formatted')

# AFTER: re-cache and materialise
spark.catalog.cacheTable(f'{CATALOG}.silver.events_clean')
_ = spark.table(f'{CATALOG}.silver.events_clean').count()
print('=== AFTER CACHING — reads from memory ===')
print('Look for: InMemoryTableScan instead of FileScan')
q.explain('formatted')

# Clean up — always clear cache when done
spark.catalog.clearCache()
print('Cache cleared. Day 10 complete!')


## Day 10 Complete!

| Cell | What it does |
|------|--------------|
| 1 | Imports |
| 2 | **Task 1** — Heavy query baseline (no cache), record `t1` |
| 3 | **Task 2** — `.explain('formatted')` before caching |
| 4 | Interpret the plan in plain English |
| 5 | Partition pruning demo — filtered vs unfiltered plan |
| 6 | **Task 3** — `spark.catalog.cacheTable()` + verify with `.explain()` |
| 7 | **Task 4** — Same query with cache, compare `t1` vs `t2` |
| 8 | Run 3 times — show cold vs warm cache timing |
| 9 | Before/after plan side by side + cache cleanup |

**Key rule:** Always call `spark.catalog.clearCache()` at the end to free cluster memory.